# Conventional Stacking K-Fold After Tuning (7 Models)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    StackingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from imblearn.combine import SMOTEENN

RANDOM_STATE = 0
N_SPLITS = 5
TARGET_COL = "HeartDisease"

DATA_PATH = "../Data/Partition_class0_cluster5_cleaned_encoded_standardized.csv"


In [ ]:
#Load dataset
df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]


In [ ]:
def lr_model(**params):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(random_state=RANDOM_STATE, **params)
    )

def svm_model(**params):
    return make_pipeline(
        StandardScaler(),
        SVC(probability=True, random_state=RANDOM_STATE, **params)
    )

def rf_model(**params):
    return make_pipeline(
        StandardScaler(),
        RandomForestClassifier(random_state=RANDOM_STATE, **params)
    )

def et_model(**params):
    return make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(random_state=RANDOM_STATE, **params)
    )

def build_stacking_model(config):
    """Build a conventional stacking model from a configuration dictionary."""
    base_estimators = []

    for learner_name, learner_params in config["base_learners"]:
        if learner_name == "LR":
            base_estimators.append(("LR", lr_model(**learner_params)))
        elif learner_name == "SVM":
            base_estimators.append(("SVM", svm_model(**learner_params)))
        elif learner_name == "RF":
            base_estimators.append(("RF", rf_model(**learner_params)))
        elif learner_name == "ET":
            base_estimators.append(("ET", et_model(**learner_params)))
        else:
            raise ValueError(f"Unknown learner: {learner_name}")

    meta_learner = LogisticRegression(
        random_state=RANDOM_STATE,
        **config["meta_learner"]
    )

    return StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_learner,
        cv=5,
        passthrough=False,
        n_jobs=-1
    )


def evaluate_model_kfold(model_name, config, X, y, n_splits=N_SPLITS):
    """Evaluate one stacking configuration using stratified K-fold cross-validation."""
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    fold_metrics = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train, y_train)

        model = build_stacking_model(config)
        model.fit(X_train_res, y_train_res)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        cm = confusion_matrix(y_test, y_pred)

        fold_metrics.append({
            "Model": model_name,
            "Fold": fold_idx,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1_score": f1_score(y_test, y_pred, zero_division=0),
            "AUC": roc_auc_score(y_test, y_proba),
            "TN": int(cm[0, 0]),
            "FP": int(cm[0, 1]),
            "FN": int(cm[1, 0]),
            "TP": int(cm[1, 1]),
        })

    return pd.DataFrame(fold_metrics)


def summarize_results(results_df):
    """Summarize mean and standard deviation across folds."""
    metrics = ["Accuracy", "Precision", "Recall", "F1_score", "AUC"]

    summary = (
        results_df
        .groupby("Model")[metrics]
        .agg(["mean", "std"])
        .reset_index()
    )

    summary.columns = [
        "_".join(col).strip("_") if isinstance(col, tuple) else col
        for col in summary.columns
    ]

    return summary.sort_values("Recall_mean", ascending=False).reset_index(drop=True)


In [ ]:
# ============================================================
# 4. After-tuning configurations for 7 conventional stacking models
# ============================================================

STACKING_CONFIGS = {
    "LR+ET": {
        "base_learners": [
            ("LR", {
                "C": 0.00013425445613221055,
                "max_iter": 1560,
            }),
            ("ET", {
                "n_estimators": 122,
                "max_depth": 26,
                "min_samples_split": 12,
                "min_samples_leaf": 3,
                "max_features": "log2",
                "bootstrap": True,
            }),
        ],
        "meta_learner": {
            "C": 0.0001018919660852145,
            "max_iter": 1815,
        },
    },

    "RF+ET": {
        "base_learners": [
            ("RF", {
                "n_estimators": 76,
                "max_depth": 22,
                "min_samples_split": 3,
                "min_samples_leaf": 10,
                "max_features": "log2",
                "bootstrap": False,
            }),
            ("ET", {
                "n_estimators": 94,
                "max_depth": 45,
                "min_samples_split": 16,
                "min_samples_leaf": 8,
                "max_features": None,
                "bootstrap": False,
            }),
        ],
        "meta_learner": {
            "C": 0.00010792177277936667,
            "max_iter": 499,
        },
    },

    "SVM+ET": {
        "base_learners": [
            ("SVM", {
                "C": 0.030695155161630694,
                "kernel": "linear",
                "gamma": "scale",
            }),
            ("ET", {
                "n_estimators": 106,
                "max_depth": 3,
                "min_samples_split": 5,
                "min_samples_leaf": 9,
                "max_features": "sqrt",
                "bootstrap": False,
            }),
        ],
        "meta_learner": {
            "C": 0.010301783397406505,
            "max_iter": 959,
        },
    },

    "RF+LR+ET": {
        "base_learners": [
            ("RF", {
                "n_estimators": 211,
                "max_depth": 48,
                "min_samples_split": 2,
                "min_samples_leaf": 9,
                "max_features": "log2",
                "bootstrap": True,
            }),
            ("LR", {
                "C": 3.339643104354595,
                "max_iter": 655,
            }),
            ("ET", {
                "n_estimators": 228,
                "max_depth": 47,
                "min_samples_split": 14,
                "min_samples_leaf": 9,
                "max_features": "sqrt",
                "bootstrap": True,
            }),
        ],
        "meta_learner": {
            "C": 0.00010165324663970105,
            "max_iter": 348,
        },
    },

    "RF+SVM+ET": {
        "base_learners": [
            ("RF", {
                "n_estimators": 136,
                "max_depth": 15,
                "min_samples_split": 9,
                "min_samples_leaf": 8,
                "max_features": "sqrt",
                "bootstrap": True,
            }),
            ("SVM", {
                "C": 0.5547119471592125,
                "kernel": "linear",
                "class_weight": None,
            }),
            ("ET", {
                "n_estimators": 104,
                "max_depth": 28,
                "min_samples_split": 10,
                "min_samples_leaf": 10,
                "max_features": "sqrt",
                "bootstrap": False,
            }),
        ],
        "meta_learner": {
            "C": 0.010301783397406505,
            "max_iter": 959,
        },
    },

    "LR+SVM+ET": {
        "base_learners": [
            ("LR", {
                "penalty": "l1",
                "solver": "saga",
                "C": 0.010289782695947526,
                "max_iter": 20,
            }),
            ("SVM", {
                "C": 3.30695155161630694,
                "kernel": "linear",
                "gamma": "auto",
            }),
            ("ET", {
                "n_estimators": 144,
                "max_depth": 18,
                "min_samples_split": 10,
                "min_samples_leaf": 10,
                "max_features": "sqrt",
                "bootstrap": False,
            }),
        ],
        "meta_learner": {
            "C": 0.00010301783397406505,
            "max_iter": 959,
        },
    },

    "RF+LR+SVM+ET": {
        "base_learners": [
            ("RF", {
                "n_estimators": 136,
                "max_depth": 15,
                "min_samples_split": 9,
                "min_samples_leaf": 8,
                "max_features": "sqrt",
                "bootstrap": True,
            }),
            ("LR", {
                "penalty": "l1",
                "solver": "saga",
                "C": 0.010289782695947526,
                "max_iter": 20,
            }),
            ("SVM", {
                "C": 0.30695155161630694,
                "kernel": "linear",
                "class_weight": None,
            }),
            ("ET", {
                "n_estimators": 104,
                "max_depth": 28,
                "min_samples_split": 10,
                "min_samples_leaf": 10,
                "max_features": "sqrt",
                "bootstrap": False,
            }),
        ],
        "meta_learner": {
            "C": 0.0010301783397406505,
            "max_iter": 959,
        },
    },
}
